# JN-L — Fiscal Flows: what new housing pays the city, and what the city's money flows look like

**The question (John, 2026-08-09):** how much new property tax does a housing CO generate? How much do
ADUs generate? How is the property-tax dollar split between Berkeley, Alameda County, and the State —
and how do the city's revenue streams (property / sales / transfer taxes, developer + in-lieu fees) flow?
And what funds actually built Berkeley's housing projects?

**Every quantitative figure here is DERIVED** from the canonical database, opened read-only, and
asserted against an external timestamped baseline (`data/baselines/fiscal_flows_baseline_*_v4.json`):
- `berkeley_housing_v4.db` — the serving view `v_projects` (completions, units, permit valuations, the
  City's own Fees-tab totals, affordability evidence, the funding ledger) and the Alameda assessor values
  v4 carries (`assessed_values`, provenance in its `sources` table).

**Re-pointed from v2 on 2026-10-02.** Earlier runs read `berkeley_housing_v2.db`, whose fee layer was a
single v1-era "total paid" figure for 57 projects ($14.1M). v4 holds **the City's own CapDetail Fees-tab
totals** for every harvested master building permit and for the projects' Planning records — a far
wider layer, but **TOTALS, not itemized**: we can say what was paid on a permit, not what each fee was
for (impact fee vs plan check vs in-lieu). §8 says so where it matters. v2-era baselines are kept
unedited; §10 gates against a v4-era baseline only.

**External-facts layer.** City budget-document figures (the FY27 GF revenue mix, the 32.57% AB-8 city
share, Measure O) are recorded IN THE BASELINE with provenance URLs and rendered as *context* — we never
derive our numbers from them. CKAN appears nowhere in this notebook.

**Standing caveats (read before trusting any figure):**
1. **Reassessment lag** — the County enrolls new construction 1–2 years after CO; every "enrolled" figure
   for 2024–2026 completions is a KNOWN UNDERSTATEMENT (this is measured in §3, not assumed).
2. **Declared permit valuations understate** — the assessor enrolls market value of new construction;
   declared construction cost is a floor.
3. **Completed projects with no assessor row** are counted in §2 and their value EXCLUDED — another
   understatement (re-platted / too-new parcels; the CLAUDE.md standing guard).

## §1 — Setup: read-only connection, sha pins, the baseline
**Where from.** v4 opened read-only (`mode=ro`). The assessor values are the snapshot v4 ingested; its
`sources` row carries the as-of date and a content checksum, which the baseline pins alongside the v4
file sha — a drifted pin means you are estimating from a different world and the gate (§10) will say so.

In [1]:
import os, sys, sqlite3, json, glob, hashlib, statistics
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook_connected'   # so plotly figures survive nbconvert-to-HTML export
ROOT = os.path.expanduser('~/berkeley-data')
V4 = os.path.join(ROOT, 'databases', 'berkeley_housing_v4.db')
SMALL_MAX_UNITS = 2   # <=2 units: declared valuation, not parcel Imps (§4)
COMPLETED_SQL = '''
SELECT p.project_id, p.project_key, p.units_built_or_permitted AS units, p.adu_units,
       p.co_issued_date, p.rhna_excluded, p.construction_valuation AS valuation,
       p.fees_paid, p.fees_outstanding, p.planning_fees_paid, p.vli_units,
       (SELECT SUM(a.improvements) FROM project_parcels pp JOIN assessed_values a USING(parcel_id)
         WHERE pp.project_id = p.project_id) AS imps,
       (SELECT COUNT(*) FROM project_parcels pp JOIN assessed_values a USING(parcel_id)
         WHERE pp.project_id = p.project_id) AS n_av
FROM v_projects p WHERE p.co_issued_date IS NOT NULL'''

def ro(p): return sqlite3.connect(f'file:{p}?mode=ro', uri=True)
def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

if not os.path.exists(V4):
    print("=" * 74)
    print("JN-L is an INVESTIGATION notebook: it derives everything live from the")
    print("project's local database, which is not distributed publicly:")
    print("   missing:", V4)
    print()
    print("On Colab/GitHub this notebook is READ-ONLY — reopen the link (without")
    print("running) to see the saved charts and prose.")
    print("For a notebook that RUNS anywhere, start with curriculum module FIN1:")
    print("https://colab.research.google.com/drive/1tc0oJSf4b5Iv2rNkLz3nAIk947W7LUvA")
    print("=" * 74)
    raise SystemExit("JN-L: local database required — stopping cleanly (not an error).")

BL_DIR = os.environ.get('JN_BASELINE_DIR') or os.path.join(ROOT, 'data', 'baselines')
_all = sorted(glob.glob(os.path.join(BL_DIR, 'fiscal_flows_baseline_*.json')))
_v4 = [f for f in _all if json.load(open(f)).get('db') == 'v4']
BASE = json.load(open(_v4[-1])) if _v4 else None
# external facts are the same document-sourced layer in every baseline; read the newest available
EXT_SRC = json.load(open((_v4 or _all)[-1]))
ext = EXT_SRC['external_facts']
con = ro(V4)
src = con.execute('''SELECT source_id, source_asof, retrieved_at, substr(checksum,1,16) FROM sources
                     WHERE source_kind='assessor' ORDER BY retrieved_at DESC LIMIT 1''').fetchone()
v4_now = file_sha(V4)
print('v4-era baseline:', os.path.basename(_v4[-1]) if _v4 else 'NONE YET (§10 will halt)')
if BASE:
    print(f"v4 sha now={v4_now} baseline={BASE['v4_sha']} {'MATCH' if v4_now == BASE['v4_sha'] else 'DRIFT — figures may differ'}")
    print(f"assessor checksum now={src[3]} baseline={BASE['assessor_checksum16']} "
          f"{'MATCH' if src[3] == BASE['assessor_checksum16'] else 'DRIFT'}")
print(f'assessor values in v4: source {src[0]}, as-of {src[1]}, ingested {src[2][:10]}')

v4-era baseline: fiscal_flows_baseline_2026-10-03_v4.json
v4 sha now=dcf247fcee6c9b1b baseline=dcf247fcee6c9b1b MATCH
assessor checksum now=96aa115db20517d4 baseline=96aa115db20517d4 MATCH
assessor values in v4: source 1307, as-of 2026-02-01, ingested 2026-10-03


## §2 — The completed-housing universe and its cohorts
**Assumption + plan.** Every project with a `co_issued_date` in `v_projects` (all of its buildings
finaled — v4's completion signal, built from the buildings' master-permit events). Three cohorts:
**UC / outside city permitting** (`rhna_excluded`: Regents land, off the tax roll — no city property-tax
increment); **small** (≤2 units — ADUs and small infill); **tracked** (≥3 units).
**Why the split matters fiscally:** for a tracked project the parcel's improvements ARE (mostly) the new
building; for an ADU the parcel's improvements are DOMINATED by the pre-existing house that was already
taxed — using parcel Imps for ADUs would massively overstate the new-tax increment.
**Change from v2:** v2 split by an id-block heuristic (ids 185–899 = the CO-only import cohort); v4 has
no such block, so the split is by the units the buildings actually carry.

In [2]:
con.row_factory = sqlite3.Row
rows = con.execute(COMPLETED_SQL).fetchall()
completed_projects = len(rows)
completed_units = sum(r['units'] or 0 for r in rows)
matched = sum(1 for r in rows if r['n_av'])
uc = [r for r in rows if r['rhna_excluded']]
small = [r for r in rows if not r['rhna_excluded'] and (r['units'] or 0) <= SMALL_MAX_UNITS]
tracked = [r for r in rows if not r['rhna_excluded'] and (r['units'] or 0) > SMALL_MAX_UNITS]
tracked_imps = 0.0; tracked_units = 0; per_year = {}; bench = []
for r in tracked:
    u = r['units'] or 0; imps = r['imps'] or 0.0
    tracked_imps += imps; tracked_units += u
    yr = r['co_issued_date'][:4]
    per_year.setdefault(yr, [0, 0.0]); per_year[yr][0] += u; per_year[yr][1] += imps
    if u >= 100 and imps > 0: bench.append((imps/u, r['project_key']))
co_years = sorted(r['co_issued_date'][:4] for r in rows)
print(f'completed: {completed_projects} projects / {completed_units:,} units (COs {co_years[0]}–{co_years[-1]})')
print(f'cohorts: UC/outside-city {len(uc)} | small (<={SMALL_MAX_UNITS}u) {len(small)} | tracked (>={SMALL_MAX_UNITS+1}u) {len(tracked)}')
print(f'assessor-linked: {matched} of {completed_projects} ({completed_projects-matched} with no assessor row = re-plat/stale-APN/too-new, EXCLUDED)')
print(f'tracked cohort: {tracked_units:,} units; enrolled improvements ${tracked_imps/1e6:.1f}M')

completed: 888 projects / 4,462 units (COs 2018–2026)
cohorts: UC/outside-city 1 | small (<=2u) 824 | tracked (>=3u) 63
assessor-linked: 885 of 888 (3 with no assessor row = re-plat/stale-APN/too-new, EXCLUDED)
tracked cohort: 3,578 units; enrolled improvements $1615.2M


**Found + verify.** The unlinked remainder is the standing stale-APN class — excluded, so totals lean
conservative. The gate (§10) pins the project/unit/link counts. Note that v4 counts projects as
*application-anchored or building-anchored* units of work: a lot with a house and a later ADU on separate
permits can be two completed projects sharing a parcel — the parcel's Imps then count once per project
in the tracked sum only if both are ≥3 units (rare).

## §3 — Enrolled value vs. the lag: what the assessor has actually posted
**Assumption + plan.** Sum enrolled improvement value (`Imps`) for tracked projects by CO year, compute
$/unit, and compare against the *benchmark*: the **median** Imps/unit among large (≥100-unit) completed
projects (John, 2026-10-03). The single most-enrolled building is shown as the high end: it was the old
benchmark, but the largest is now a student building whose units hold several beds each.

📝 **What to read from the chart:** enrolled $/unit collapses for recent CO years — that is the
County's reassessment lag made visible, not a collapse in building value.

In [3]:
import plotly.graph_objects as go
bench_per_unit = statistics.median(sorted(v for v, _ in bench)); bench_key = f'median of {len(bench)}'
high_per_unit, high_key = max(bench)
py = pd.DataFrame([(y, v[0], v[1]/1e6, (v[1]/v[0]/1e3 if v[0] else 0)) for y, v in sorted(per_year.items())],
                  columns=['co_year', 'units', 'enrolled_imps_M', 'enrolled_k_per_unit'])
fig = go.Figure()
fig.add_bar(x=py.co_year, y=py.enrolled_k_per_unit, name='enrolled $K/unit (derived)', marker_color='#7ec8e3')
fig.add_scatter(x=py.co_year, y=[bench_per_unit/1e3]*len(py), mode='lines',
                name=f'benchmark ${bench_per_unit/1e3:.0f}K/unit ({bench_key} large buildings)',
                line=dict(color='#ffd166', dash='dash'))
fig.add_scatter(x=py.co_year, y=[high_per_unit/1e3]*len(py), mode='lines',
                name=f'high end ${high_per_unit/1e3:.0f}K/unit ({high_key}, most enrolled)',
                line=dict(color='#ef8354', dash='dot'))
fig.update_layout(template='plotly_dark', height=380,
                  title='Tracked completions: enrolled assessed $/unit by CO year vs full-enrollment benchmark (derived §3)',
                  yaxis_title='$K per unit')
fig.show()
print(py.to_string(index=False))
print(f'benchmark: {bench_key} at ${bench_per_unit:,.0f}/unit; high end {high_key} at ${high_per_unit:,.0f}/unit')
_bn = sorted(v for v, _ in bench)
print(f'context (not gated): {len(_bn)} tracked >=100u projects with Imps; median ${statistics.median(_bn):,.0f}/unit, '
      f'max/median {max(_bn)/statistics.median(_bn):.1f}x')

co_year  units  enrolled_imps_M  enrolled_k_per_unit
   2018    166        74.414662           448.281096
   2019    213       100.541844           472.027437
   2020    322       226.844226           704.485174
   2021    301       196.815871           653.873326
   2022    607       261.500484           430.808046
   2023    643       353.265105           549.401407
   2024    608       216.897791           356.739788
   2025    444       140.871218           317.277518
   2026    274        44.059361           160.800588
benchmark: median of 9 at $440,790/unit; high end ZP2017-0083 at $921,527/unit
context (not gated): 9 tracked >=100u projects with Imps; median $440,790/unit, max/median 2.1x


📝 **What this chart could mislead about.** (1) The benchmark is ONE building — unit mix, parking, and
construction vintage vary; treat it as a central estimate, not a law. (2) Early years sit low partly from
older, cheaper construction and condo-APN linkage gaps — not purely lag. (3) The y-axis is $/unit, not
total value: read the printed `units` column for each bar's width in homes. (4) **The max-rule is
sensitive to what a "unit" is.** v4 links far more completed projects to their parcels than v2 did, and
the most-enrolled large building is now a student building whose units hold several beds each — a high
$/unit that is real for that building but not typical. The printed median is the robustness check; a
change of rule (median, or $/sq ft) is a methodology decision, not drift (open for John, 2026-10-02).

## §4 — The ADU / small-infill increment: declared valuations, not parcel values
**Assumption + plan.** For completed small projects (≤2 units, not UC), the new-tax increment is the
*declared building-permit valuation* (`construction_valuation` = the master permits' CPRA `JobValuation`)
— the parcel's Imps would count the main house that was already taxed. Declared cost is a FLOOR: the
assessor enrolls market value of the addition. Separately, the CPRA permit's own **ADU** field gives the
count of ADU units among all completions.

In [4]:
vals = [r['valuation'] for r in small if (r['valuation'] or 0) > 0]
adu_small_n = len(vals)
adu_small_median = statistics.median(vals)
adu_small_sum = sum(vals)
adu_flag_units = sum(r['adu_units'] or 0 for r in rows)
print(f'small (<={SMALL_MAX_UNITS}u) completions with a declared valuation: n={adu_small_n} of {len(small)}')
print(f'declared valuation: median ${adu_small_median:,.0f}, mean ${adu_small_sum/adu_small_n:,.0f}, sum ${adu_small_sum/1e6:.1f}M')
print(f'ADU units among ALL completions (CPRA ADU field on the master permit): {adu_flag_units:,}')

small (<=2u) completions with a declared valuation: n=824 of 824
declared valuation: median $130,000, mean $188,019, sum $154.9M
ADU units among ALL completions (CPRA ADU field on the master permit): 623


**Found + verify.** The median declared small-project valuation is the load-bearing per-ADU figure
(gated §10). A ≤2-unit project is not always an ADU (a new single-family house or duplex qualifies too),
which is why the ADU-field count is printed separately rather than used as the cohort.

## §5 — The tax math: per-unit rules of thumb and the city's slice
**Assumption + plan.** New assessed value × the 1% AB-8 levy × the city's 32.57% share = the General-Fund
increment; the ~0.23% voter-debt levy flows to GO debt-service funds (NOT the GF). Rates and shares come
from the baseline's external-facts layer (with provenance) — the cells read them, never literals.

In [5]:
city_share = ext['city_share_of_1pct']['value']
tot_rate  = ext['ad_valorem_total_rate']['value']
mf_av_per_unit = bench_per_unit                       # derived §3
adu_av = adu_small_median                             # derived §4 (declared floor)
rules = pd.DataFrame({
    'new multifamily unit': [mf_av_per_unit, mf_av_per_unit*tot_rate, mf_av_per_unit*0.01*city_share],
    'new ADU (declared floor)': [adu_av, adu_av*tot_rate, adu_av*0.01*city_share]},
    index=['new assessed value ($)', f'total ad-valorem @ {tot_rate:.2%} ($/yr)', 'City GF share ($/yr)'])
print(rules.round(0).to_string())

small_units = sum(r['units'] or 0 for r in small)
mf_units = tracked_units
enrolled_now = tracked_imps + adu_small_sum
full_av = mf_units * mf_av_per_unit + adu_small_sum
print(f'\ntracked units {mf_units:,} × ${mf_av_per_unit/1e3:.0f}K + small-cohort ${adu_small_sum/1e6:.0f}M declared ({small_units:,} units)')
print(f'ENROLLED NOW (floor): ${enrolled_now/1e6:.0f}M → total 1% levy ${enrolled_now*0.01/1e6:.2f}M/yr → City GF ${enrolled_now*0.01*city_share/1e6:.2f}M/yr')
print(f'FULL ENROLLMENT (est): ${full_av/1e9:.2f}B → total 1% levy ${full_av*0.01/1e6:.1f}M/yr → City GF ${full_av*0.01*city_share/1e6:.1f}M/yr')

                                 new multifamily unit  new ADU (declared floor)
new assessed value ($)                       440790.0                  130000.0
total ad-valorem @ 1.25% ($/yr)                5510.0                    1625.0
City GF share ($/yr)                           1436.0                     423.0

tracked units 3,578 × $441K + small-cohort $155M declared (884 units)
ENROLLED NOW (floor): $1770M → total 1% levy $17.70M/yr → City GF $5.77M/yr
FULL ENROLLMENT (est): $1.73B → total 1% levy $17.3M/yr → City GF $5.6M/yr


**Found + verify.** Two figures bracket reality: the *enrolled-now floor* (what the County has posted)
and the *full-enrollment estimate* (units × benchmark). The truth converges toward the estimate as the
County works its 1–2-year backlog. **Mislead check:** the estimate multiplies the tracked units by ONE
benchmark building's $/unit — a ±15% band is honest; and the small-cohort term uses declared cost (floor).

## §6 — Flow of city revenues (Sankey 1): the FY27 General Fund, external-facts layer
📝 **Before.** This is the *context* diagram — Berkeley's PROPOSED FY27 GF revenue mix flowing into the
GF and out to departments, drawn from the baseline's external-facts (May-14-2026 budget presentation,
provenance in the baseline JSON). Permit fees are drawn to DEDICATED funds — they never enter the GF,
which is exactly why they can't help the $30M operating deficit. The fee link is DERIVED: the average
annual building-permit fees paid on projects whose first BP was issued 2018–2025 (v4 Fees-tab totals).

In [6]:
fee_by_bp_year = con.execute('''SELECT substr(bp_issued_date,1,4) y, SUM(fees_paid) FROM v_projects
    WHERE fees_paid IS NOT NULL AND substr(bp_issued_date,1,4) BETWEEN '2018' AND '2025' GROUP BY 1''').fetchall()
bp_fee_per_year_m = sum(v for _, v in fee_by_bp_year) / 8 / 1e6
rev = ext['gf_revenues_fy27_m']['value']; spend = ext['gf_spend_fy27_m']['value']
labels = ['Property tax (1% share)', 'Transfer taxes', 'Business license', 'Sales + soda tax',
          'Utility users tax', 'Other GF revenue', 'GENERAL FUND',
          'Police', 'Fire', 'Non-Departmental', 'HHCS', 'City Manager', 'Parks Rec Waterfront', 'All other depts',
          'Building-permit fees on housing (avg/yr, derived)', 'Dedicated funds (NOT GF)']
GF = 6
src_ = [0,1,2,3,4,5, GF,GF,GF,GF,GF,GF,GF, 14]
dst = [GF]*6 + [7,8,9,10,11,12,13] + [15]
val = [rev['property_tax'], rev['transfer_taxes'], rev['business_license'], rev['sales_and_soda'],
       rev['utility_users'], rev['other'],
       spend['Police'], spend['Fire'], spend['Non-Departmental'], spend['HHCS'],
       spend['City Manager'], spend['Parks Rec Waterfront'], spend['All other'],
       bp_fee_per_year_m]
fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=12, thickness=14,
              color=['#7ec8e3']*6 + ['#ffd166'] + ['#ef8a62']*7 + ['#b3b3ff'] + ['#8dd3c7']),
    link=dict(source=src_, target=dst, value=val)))
fig.update_layout(template='plotly_dark', height=520,
                  title='Berkeley FY27 General Fund flows ($M, PROPOSED budget — external facts) + housing BP fees to dedicated funds')
fig.show()
print('GF revenue total $%.1fM / spend total $%.1fM (proposed FY27; source in baseline external_facts)' % (rev['total'], spend['total']))
print(f'housing building-permit fees: ${bp_fee_per_year_m:.1f}M/yr average over BP years 2018-2025 (v4 Fees-tab totals)')
print('in-lieu / Housing Trust Fund / inclusionary fees: NOT MEASURED in v4 -- no link drawn (unknown with provenance)')

GF revenue total $314.9M / spend total $313.9M (proposed FY27; source in baseline external_facts)
housing building-permit fees: $3.6M/yr average over BP years 2018-2025 (v4 Fees-tab totals)
in-lieu / Housing Trust Fund / inclusionary fees: NOT MEASURED in v4 -- no link drawn (unknown with provenance)


📝 **What this Sankey could mislead about.** (1) These are PROPOSED FY27 figures, not actuals. (2) The fee
link is an 8-year *average* of totals paid as of retrieval, assigned to the project's first-BP year — fees
are paid over a permit's life, so it is a scale, not a cash-flow year. (3) In-lieu / impact fees are
**absent, not zero**: the Fees tab gives totals per permit, not fee types, so we cannot separate an
impact fee from plan-check — the diagram must not be read as "no in-lieu money". (4) "Non-Departmental"
hides debt service, insurance, and transfers — it is not a service.

## §7 — Where a new-housing property-tax dollar goes (Sankey 2): the derived increment
📝 **Before.** The §5 full-enrollment estimate flowing through the AB-8 split (city share exact at
32.57%; schools/county/special approximate — flagged in the baseline). The State's box is drawn at ZERO
by construction: property tax never leaves the county; ERAF's school dollars *relieve* the State's
Prop-98 obligation instead.

In [7]:
alloc = ext['allocation_approx']['value']
levy = full_av * 0.01 / 1e6          # $M/yr at full enrollment (derived §5)
debt = full_av * ext['city_debt_levy_rate']['value'] / 1e6
mf_share  = (mf_units * mf_av_per_unit) / full_av
labels2 = [f'New multifamily AV (~{mf_units:,}u)', f'New small/ADU AV ({adu_small_n} projects, declared)',
           '1% AB-8 levy', 'City of Berkeley GF (32.57%)', 'Schools + ERAF (~45%)',
           'Alameda County (~15%)', 'Special districts (~7%)', 'Voter debt levies (GO bonds)',
           'State of California ($0 direct)']
src2 = [0, 1, 2, 2, 2, 2, 0, 1]
dst2 = [2, 2, 3, 4, 5, 6, 7, 7]
val2 = [levy*mf_share, levy*(1-mf_share),
        levy*alloc['city'], levy*alloc['schools_incl_eraf'], levy*alloc['county'], levy*alloc['special_districts'],
        debt*mf_share, debt*(1-mf_share)]
fig = go.Figure(go.Sankey(
    node=dict(label=labels2, pad=12, thickness=14,
              color=['#7ec8e3', '#8dd3c7', '#ffd166', '#ef8a62', '#b3b3ff', '#b3b3ff', '#b3b3ff', '#cccccc', '#555555']),
    link=dict(source=src2, target=dst2, value=val2)))
fig.update_layout(template='plotly_dark', height=460,
                  title=f'Annual property tax from housing completed {co_years[0][:4]}–{co_years[-1][:4]} at FULL enrollment (est ${levy:.1f}M/yr levy + ${debt:.1f}M/yr debt levies)')
fig.show()
print(f"City GF increment at full enrollment: ${levy*alloc['city']:.1f}M/yr — vs the ~$30M structural deficit")

City GF increment at full enrollment: $5.6M/yr — vs the ~$30M structural deficit


📝 **What this Sankey could mislead about.** (1) DO NOT compare its widths to Sankey 1 visually — this
whole diagram is a small fraction of the GF diagram above. (2) It shows the FULL-ENROLLMENT estimate, not
current cash — today's enrolled floor (§5) is well below it. (3) The schools/county/special split is
approximate (baseline flags it); only the 32.57% city share is document-exact. (4) The State box at $0 is
*direct* incidence — via ERAF the State budget benefits indirectly.

## §8 — Fees paid, and the sources of funds that BUILT the housing (what v4 can and cannot say)
**Assumption + plan.** Three layers, each named for what it is:
(a) **building-permit fees** — the City's own CapDetail *Fees tab* total for each master/phase permit
of a project's buildings, summed per project, **as of retrieval** (`fees_paid`, `fees_outstanding`,
`fees_as_of`); (b) **Planning fees** — the same, on the project's Planning records
(`planning_fees_paid`); (c) the **funding ledger** v4 keys to projects (Council resolutions, CTCAC and
HCD awards) and affordability evidence on completed units.
**What this layer is NOT:** itemized. The Fees tab shows totals and the first page of invoices; it does
not tell an impact fee from plan check, and in-lieu / Housing Trust Fund payments that are not invoiced
on a permit are not in it. Where v4 is silent the answer is **unknown with provenance** — never filled
from an oracle.

In [8]:
fees_df = pd.read_sql('''SELECT project_key, name, status, units_built_or_permitted units, bp_issued_date,
                                fees_paid, fees_outstanding, fees_as_of, planning_fees_paid FROM v_projects''', con)
by_status = (fees_df.groupby('status')
             .agg(projects_with_bp_fees=('fees_paid', 'count'), bp_fees_paid=('fees_paid', 'sum'),
                  projects_with_planning_fees=('planning_fees_paid', 'count'), planning_fees_paid=('planning_fees_paid', 'sum'))
             .sort_values('bp_fees_paid', ascending=False))
bp_fee_projects = int(fees_df.fees_paid.notna().sum())
bp_fees_paid = float(fees_df.fees_paid.sum())
bp_fees_outstanding = float(fees_df.fees_outstanding.sum())
planning_fee_projects = int(fees_df.planning_fees_paid.notna().sum())
planning_fees_paid = float(fees_df.planning_fees_paid.sum())
bp_fees_paid_completed = sum(r['fees_paid'] or 0 for r in rows)
print('FEE RECORD — the City\'s own Fees-tab TOTALS (not itemized), as of', fees_df.fees_as_of.max())
print(f'   building permits: ${bp_fees_paid/1e6:.1f}M paid across {bp_fee_projects} projects '
      f'(${bp_fees_outstanding:,.0f} outstanding); completed projects ${bp_fees_paid_completed/1e6:.1f}M')
print(f'   Planning records: ${planning_fees_paid/1e6:.1f}M paid across {planning_fee_projects} projects')
print(by_status.round(0).to_string())
per_unit = fees_df[(fees_df.units >= 5) & fees_df.fees_paid.notna()].assign(per_unit=lambda d: d.fees_paid / d.units)
print(f'\n   BP fees per unit, 5+ unit projects with fees (n={len(per_unit)}): median ${per_unit.per_unit.median():,.0f}, '
      f'IQR ${per_unit.per_unit.quantile(.25):,.0f}–${per_unit.per_unit.quantile(.75):,.0f}')
print('   largest:')
for _, r in fees_df.sort_values('fees_paid', ascending=False).head(8).iterrows():
    print(f"     {r.project_key:14} {str(r['name'])[:34]:34} {int(r.units or 0):>4}u  ${r.fees_paid:>12,.0f}")

vli = sum(r['vli_units'] or 0 for r in rows)
db_projects = con.execute("SELECT COUNT(*) FROM v_projects WHERE pathways LIKE '%density_bonus%'").fetchone()[0]
awards = {}
for key, status, f in con.execute('SELECT project_key, status, funding FROM v_projects WHERE funding IS NOT NULL'):
    for x in json.loads(f):
        fid = x.get('funding_id') or f'{key}:{len(awards)}'
        try: amt = float(x.get('amount_usd') or 0)
        except ValueError: amt = 0.0
        awards.setdefault(fid, dict(funder=x.get('funder'), program=x.get('program'), amount=amt, projects=set()))['projects'].add(key)
funding_projects = len(set().union(*[a['projects'] for a in awards.values()])) if awards else 0
funding_total = sum(a['amount'] for a in awards.values())
print('\nFUNDING LEDGER keyed to projects (deduplicated on funding_id):')
fund = (pd.DataFrame([dict(funder=a['funder'], amount=a['amount']) for a in awards.values()])
        .groupby('funder').amount.agg(['count', 'sum']).sort_values('sum', ascending=False))
for k, r in fund.iterrows():
    print(f"   {k[:60]:60} {int(r['count']):>3} awards  ${r['sum']:>14,.0f}")
print(f'   TOTAL ${funding_total/1e6:.1f}M across {len(awards)} awards on {funding_projects} projects')
print()
print('AFFORDABILITY / MECHANISM evidence:')
print(f'   VLI units on completed projects (affordability evidence): {vli}')
print(f'   projects using the State Density Bonus (pathways, all stages): {db_projects}')
mo = ext['measure_o']['value']
print('\nEXTERNAL-FACTS layer (provenance in baseline):')
print(f"   Measure O (2018): ${mo['bond_m']}M GO bond; with Measure P transfer-tax ~${mo['allocated_with_p_m']}M")
print(f"   allocated, supporting {mo['units_supported']:,}+ affordable units (city staff report, Dec 2025).")

FEE RECORD — the City's own Fees-tab TOTALS (not itemized), as of 2026-10-01
   building permits: $38.8M paid across 569 projects ($20,729 outstanding); completed projects $31.4M
   Planning records: $5.0M paid across 437 projects
                    projects_with_bp_fees  bp_fees_paid  projects_with_planning_fees  planning_fees_paid
status                                                                                                  
completed                             456    31385004.0                          140           1561223.0
under_construction                     83     6316622.0                           28            359965.0
stale                                  30     1126631.0                           11             66157.0
denied                                  0           0.0                            3             16499.0
entitled                                0           0.0                          112           1948073.0
in_review                         


FUNDING LEDGER keyed to projects (deduplicated on funding_id):
   City of Berkeley                                              33 awards  $   286,790,401
   State of California (CTCAC)                                   16 awards  $    45,407,634
   State of California (Strategic Growth Council / HCD)           2 awards  $    41,854,345
   State of California (HCD)                                      7 awards  $    36,816,146
   Regional Center of the East Bay                                1 awards  $     1,500,000
   Berkeley Housing Authority                                     1 awards  $             0
   TOTAL $412.4M across 60 awards on 13 projects

AFFORDABILITY / MECHANISM evidence:
   VLI units on completed projects (affordability evidence): 8
   projects using the State Density Bonus (pathways, all stages): 97

EXTERNAL-FACTS layer (provenance in baseline):
   Measure O (2018): $135M GO bond; with Measure P transfer-tax ~$238.1M
   allocated, supporting 1,421+ affordable un

**Found + verify.** The fee layer is now City-recorded rather than v1-era: Fees-tab totals on hundreds of
projects instead of one number on 57. Two cautions travel with it: totals are **as of retrieval** (a
permit still open will keep accruing), and they are **not itemized** — "paid" mixes plan-check,
permit, and any impact fees invoiced on that permit. The funding ledger covers the subsidized projects
the City and State document (Measure O / Housing Trust Fund, LIHTC, AHSC, NPLH); market-rate equity and
construction debt are not public records, so the stack is complete only for the affordable projects.
**Publishing rule:** funders printed here are public agencies and programs; no private individual is named.

## §8b — The per-permit fee snapshot (AgencyCounter `total_fee`)
**Assumption + plan.** A **dated, frozen AgencyCounter snapshot** (`data/reference/permit_fees_*.csv`,
harvested by `scripts/harvest_permit_fees.py`) carries `total_fee` PER PERMIT, including permits that are
still *under review* — fees assessed before issuance, which §8's Fees-tab totals (issued master permits)
do not yet hold. **What to read:** the fiscal engine is the **new-construction Building Permit fee**, and
it scales with project size. The big BP fee is charged around issuance, so the large-project
**application collapse (JN-J §4b) is a leading indicator of a coming decline in this fee stream**.
The snapshot's own `project_id`/`units`/`status_label` columns were copied from v2 at harvest time and
are not used here.

In [9]:
snap = sorted(glob.glob(os.path.join(ROOT, 'data', 'reference', 'permit_fees_*.csv')))[-1]
fees = pd.read_csv(snap)
fees = fees[fees.total_fee > 0].copy()

def fee_class(r):
    d = str(r.description or '').lstrip()
    if r.record_type == 'Building Permit' and not d.startswith('Demolition'):
        return 'New-construction BP'
    if d.startswith('Demolition') or 'Demolition' in str(r.record_type):
        return 'Demolition'
    return 'Other (mech/elec/plumb/misc)'
fees['klass'] = fees.apply(fee_class, axis=1)

by_class = fees.groupby('klass').total_fee.agg(['count', 'sum']).sort_values('sum', ascending=False)
nb_sum = float(fees.loc[fees.klass == 'New-construction BP', 'total_fee'].sum())
print(f'PER-PERMIT FEE SNAPSHOT  <-  {os.path.basename(snap)}  (frozen, gated in §10)')
print(f'  {len(fees)} fee-bearing permits at {fees.address.str.strip().str.upper().nunique()} addresses  |  total ${fees.total_fee.sum():,.0f}')
print(f'  new-construction BP fees carry ${nb_sum:,.0f} = {nb_sum/fees.total_fee.sum()*100:.0f}% of the snapshot total')
print()
for k, r in by_class.iterrows():
    print(f'   {k:34} {int(r["count"]):>3} permits   ${r["sum"]:>12,.0f}')

COLOR = {'New-construction BP': '#b5371f', 'Demolition': '#c9975a', 'Other (mech/elec/plumb/misc)': '#9aa0a6'}
top = fees.sort_values('total_fee', ascending=False).head(15).iloc[::-1]
label = [f"{a[:20]} · {p}" for a, p in zip(top.address, top.permit)]
fig = go.Figure(go.Bar(
    x=top.total_fee, y=label, orientation='h',
    marker_color=[COLOR[k] for k in top.klass],
    text=[f'${v/1e3:.0f}K' for v in top.total_fee], textposition='outside',
    customdata=top[['klass', 'permit_status']].values,
    hovertemplate='%{y}<br>$%{x:,.0f}<br>%{customdata[0]} · %{customdata[1]}<extra></extra>'))
fig.update_layout(
    title=f'Per-permit city fees — top 15 charges  ({os.path.basename(snap)})',
    xaxis_title='total fee charged ($)', height=520,
    margin=dict(l=8, r=60, t=54, b=40), plot_bgcolor='white',
    annotations=[dict(x=0.98, y=0.06, xref='paper', yref='paper', showarrow=False, align='right',
                      font=dict(size=11, color='#6b6b6b'),
                      text='red = new-construction BP (scales with size) · tan = demolition · grey = other trades')])
fig.show()

PER-PERMIT FEE SNAPSHOT  <-  permit_fees_2026-09-01.csv  (frozen, gated in §10)
  131 fee-bearing permits at 60 addresses  |  total $1,762,514
  new-construction BP fees carry $1,531,890 = 87% of the snapshot total

   New-construction BP                 42 permits   $   1,531,890
   Demolition                          36 permits   $     204,543
   Other (mech/elec/plumb/misc)        53 permits   $      26,081


📝 **What this snapshot could mislead about.** (1) **`total_fee` is populated on only a MINORITY of
AgencyCounter records** — mostly active/recent permits — so it is a slice, **complementary to** §8's
Fees-tab totals and **not to be summed with them** (an issued permit can appear in both). (2) These are
amounts **charged on the permit record, not a cash-flow ledger**. (3) The chart is **truncated to the top
15**. (4) **No party names** — this layer carries no payer identity; the payer → city network needs the
CapDetail contacts v4 holds (`v_project_people`), names as the records give them.
(5) The snapshot is **dated and frozen**; a re-harvest writes a NEW dated file and appends a NEW baseline.

## §9 — Data lineage: how every figure in this JN was produced
```mermaid
flowchart LR
    A[CPRA permit productions 2026-07-07] --> V4[(berkeley_housing_v4.db)]
    C[Accela CapDetail Fees tab\nmaster permits + Planning records] --> V4
    B[Alameda assessor parcels\nvia databases/berkeley.db] --> V4
    L[funding ledger\nCouncil / CTCAC / HCD awards] --> V4
    V4 -->|v_projects completions| S2[§2 cohorts]
    V4 -->|assessed_values via project_parcels| S2
    S2 --> S3[§3 enrolled Imps + benchmark $/unit]
    V4 -->|construction_valuation| S4[§4 small-cohort declared values]
    S3 --> S5[§5 tax math]
    S4 --> S5
    X[City budget docs = EXTERNAL FACTS w/ provenance] -.-> S6[§6 GF Sankey]
    V4 -->|fees_paid by first-BP year| S6
    X -.-> S7[§7 allocation Sankey]
    S5 --> S7
    V4 -->|fees + funding + affordability| S8[§8 fees + funding sources]
    FEE[permit_fees_*.csv AgencyCounter snapshot] --> S8B[§8b per-permit fees]
    BL[(fiscal_flows_baseline_*_v4.json)] ==>|gate §10| S2
    BL ==> S8
    BL ==> S8B
    BL ==> S3
    BL ==> S4
```
Solid arrows = derivation from primary stores. Dotted = external-facts context (never derived-from).
Double arrows = the baseline gate. CKAN does not appear — nothing here touches the oracle.

## §10 — THE GATE: derived == external timestamped baseline, or HALT with a diagnosis
**Why.** A notebook that hardcodes its answers has the answer key baked in. Every load-bearing figure
above is re-derived here and asserted against the newest **v4-era** `fiscal_flows_baseline_*.json`. A
legitimate change (County posts reassessments, a Fees-tab re-harvest, a new ruling) re-passes by
APPENDING a new timestamped baseline — never by editing a number. The v2-era baselines are a different
database and are never compared here; the generator prints the v2 → v4 crosswalk.

In [10]:
derived_now = {
    'completed_projects': completed_projects,
    'completed_units': completed_units,
    'assessor_matched': matched,
    'adu_small_n': adu_small_n,
    'adu_small_median_valuation': round(adu_small_median),
    'adu_flag_units_completed': adu_flag_units,
    'vli_completed_units': vli,
    'density_bonus_projects': db_projects,
    'bp_fee_projects': bp_fee_projects,
    'planning_fee_projects': planning_fee_projects,
    'funding_awards': len(awards),
    'funding_projects': funding_projects,
    'tracked_imps_m': round(tracked_imps/1e6, 1),
    'adu_small_sum_m': round(adu_small_sum/1e6, 1),
    'benchmark_per_unit_k': round(bench_per_unit/1e3),
    'bp_fees_paid_m': round(bp_fees_paid/1e6, 1),
    'bp_fees_outstanding_k': round(bp_fees_outstanding/1e3),
    'bp_fees_paid_completed_m': round(bp_fees_paid_completed/1e6, 1),
    'planning_fees_paid_m': round(planning_fees_paid/1e6, 1),
    'funding_total_m': round(funding_total/1e6, 1),
    'itemized_fee_permits': int((fees.total_fee > 0).sum()),
    'itemized_fee_addresses': int(fees.address.str.strip().str.upper().nunique()),
    'itemized_fee_total_k': round(float(fees.total_fee.sum())/1e3),
    'itemized_bp_max_k': round(float(fees.loc[fees.klass == 'New-construction BP', 'total_fee'].max())/1e3),
}
assert BASE is not None, ('JN-L GATE HALT — no v4-era fiscal_flows baseline in ' + BL_DIR +
                          '. Review the figures above; append one with build_jn_l.py --write-baseline (John\'s call).')
failures = []
for section in ('hard_gated', 'rounded_gated'):
    for key, spec in BASE[section].items():
        exp, got = spec['value'], derived_now.get(key)
        ok = (got == exp)
        print(f"   {key:28} derived={got!s:<10} baseline={exp!s:<10} {'OK' if ok else 'FAIL'}")
        if not ok:
            failures.append(f"{key}: derived {got} != baseline {exp} — likely: {spec['what_would_change_it']}")
assert not failures, ('JN-L GATE HALT — derived != baseline:\n' + '\n'.join(failures)
                      + '\nLegitimate change => append a NEW timestamped baseline; never edit a value.')
print('\nJN-L GATE PASS — all gated figures match', BASE['as_of'], '(v4-era)')

   completed_projects           derived=888        baseline=888        OK
   completed_units              derived=4462       baseline=4462       OK
   assessor_matched             derived=885        baseline=885        OK
   adu_small_n                  derived=824        baseline=824        OK
   adu_small_median_valuation   derived=130000     baseline=130000     OK
   adu_flag_units_completed     derived=623        baseline=623        OK
   vli_completed_units          derived=8          baseline=8          OK
   density_bonus_projects       derived=97         baseline=97         OK
   bp_fee_projects              derived=569        baseline=569        OK
   planning_fee_projects        derived=437        baseline=437        OK
   funding_awards               derived=60         baseline=60         OK
   funding_projects             derived=13         baseline=13         OK
   itemized_fee_permits         derived=131        baseline=131        OK
   itemized_fee_addresses       derive